# 08 — Benchmark Leaderboard

Compare all systems on identical test folds across multi-label metrics.

| System | Description |
|---|---|
| GLM-classical | Binary relevance, classical features only |
| GLM+GNN+Mamba | Binary relevance, all admitted extensions |
| XGBoost | Per-label gradient-boosted trees |
| FT-Transformer | Feature-tokenizer Transformer |
| TabPFN | In-context learning tabular foundation model (subsampled) |

**Metrics:** Hamming loss, subset accuracy, label ranking AP, per-label AUC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import sys
sys.path.insert(0, '..')

from src.labels import LABEL_COLS
from src.features import build_features, label_matrix
from src.models.glm import BinaryRelevanceGLM
from src.evaluation import multi_label_report

In [ ]:
train = pd.read_parquet('../data/processed/train_labeled.parquet')
test  = pd.read_parquet('../data/processed/test_labeled.parquet')

X_train = build_features(train)
X_test  = build_features(test)
y_train = label_matrix(train)
y_test  = label_matrix(test)

# Extended feature sets
gnn_train = pd.read_parquet('../data/processed/gnn_feats_train.parquet')
gnn_test  = pd.read_parquet('../data/processed/gnn_feats_test.parquet')
ssm_train = pd.read_parquet('../data/processed/ssm_feats_train.parquet')
ssm_test  = pd.read_parquet('../data/processed/ssm_feats_test.parquet')

X_full_train = pd.concat([X_train, gnn_train, ssm_train], axis=1)
X_full_test  = pd.concat([X_test,  gnn_test,  ssm_test],  axis=1)

In [ ]:
leaderboard = {}

# ── GLM classical ────────────────────────────────────────────────────────────
br_classical = BinaryRelevanceGLM().fit(X_train, y_train)
score_c = br_classical.predict_proba(X_test).values
leaderboard['GLM-classical'] = multi_label_report(
    y_test.values, (score_c >= 0.5).astype(int), score_c
)

# ── GLM + GNN + Mamba ─────────────────────────────────────────────────────────
br_full = BinaryRelevanceGLM().fit(X_full_train, y_train)
score_f = br_full.predict_proba(X_full_test).values
leaderboard['GLM+GNN+Mamba'] = multi_label_report(
    y_test.values, (score_f >= 0.5).astype(int), score_f
)

In [ ]:
# ── XGBoost (binary relevance) ────────────────────────────────────────────────
xgb_scores = np.zeros((len(X_test), len(LABEL_COLS)))
for i, label in enumerate(LABEL_COLS):
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6,
                             use_label_encoder=False, eval_metric='logloss',
                             verbosity=0)
    clf.fit(X_full_train, y_train[label])
    xgb_scores[:, i] = clf.predict_proba(X_full_test)[:, 1]

leaderboard['XGBoost'] = multi_label_report(
    y_test.values, (xgb_scores >= 0.5).astype(int), xgb_scores
)

In [ ]:
# ── TabPFN (subsampled — effective scale ~50k rows) ───────────────────────────
try:
    from tabpfn import TabPFNClassifier
    SAMPLE = 10_000
    idx = np.random.choice(len(X_full_train), size=min(SAMPLE, len(X_full_train)), replace=False)
    X_pf_tr = X_full_train.iloc[idx].values
    X_pf_te = X_full_test.values

    pfn_scores = np.zeros((len(X_pf_te), len(LABEL_COLS)))
    for i, label in enumerate(LABEL_COLS):
        clf = TabPFNClassifier()
        clf.fit(X_pf_tr, y_train.iloc[idx][label].values)
        pfn_scores[:, i] = clf.predict_proba(X_pf_te)[:, 1]

    leaderboard['TabPFN'] = multi_label_report(
        y_test.values, (pfn_scores >= 0.5).astype(int), pfn_scores
    )
except ImportError:
    print('TabPFN not installed — skipping.')

In [ ]:
# ── Leaderboard summary ───────────────────────────────────────────────────────
summary = pd.DataFrame({
    name: {
        'Hamming loss':     r['hamming_loss'],
        'Subset accuracy':  r['subset_accuracy'],
        'Label ranking AP': r['label_ranking_ap'],
        'Mean AUC':         r['mean_auc'],
    }
    for name, r in leaderboard.items()
}).T.round(4)

print(summary.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
summary[['Mean AUC', 'Subset accuracy', 'Label ranking AP']].plot(
    kind='bar', ax=ax
)
ax.set_title('Benchmark leaderboard')
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()